In [ ]:
%pip install "bm25s[full]"
%pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.9/88.9 kB 7.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.3/745.3 kB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 29.2 MB/s eta 0:00:00
  Created wheel for pytrec-eval: filename=pytrec_eval-0.5-cp312-cp312-linux_x86_64.whl size=309354 sha256=734968a65152fd085351572c8ec71df26bf917e00a0c598558105a94160f7f01
  Stored in directory: /root/.cache/pip/wheels/c6/4a/9e/e17f9ea004e1c221bd0ff384732285211c4917b790d598ea51
Successfully built pytrec-eval
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 25.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1

## Instalar Bibliotecas

In [ ]:
import bm25s
import Stemmer
import json
import os
import numpy as np
import re
import subprocess
import nltk
import spacy
import time

from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from langdetect import detect, LangDetectException
from google.colab import drive
from tqdm import tqdm
from pathlib import Path


KeyboardInterrupt: 

## Baixar plugins

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'])
nlp = spacy.load('en_core_web_sm')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## Montar Drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


## Definição das Classes

In [ ]:
"""
Pipeline de pre-processamento para documentos juridicos com BM25s
Extrai paragrafos com demarcadores especiais e seus subsequentes
"""


@dataclass
class ProcessedParagraph:
    """Estrutura para armazenar paragrafo processado"""
    doc_id: str
    paragraph_id: str
    original_text: str
    cleaned_text: str
    tokens: List[str]
    processed_tokens: List[str]
    has_suppressed_marker: bool
    marker_types: List[str]
    is_french: bool


class LegalDocumentPreprocessor:
    """
    Pre-processador especializado para documentos juridicos.
    Extrai e processa paragrafos com marcadores especiais.
    """

    def __init__(self, use_spacy: bool = True):
        """
        Args:
            use_spacy: Se True, usa spaCy para tokenizacao e lematizacao.
                      Se False, usa NLTK com stemming.
        """
        self.use_spacy = use_spacy
        self.stemmer = PorterStemmer()
        self.stop_words = set(stopwords.words('english'))

        # Demarcadores especiais a buscar
        # Inclui variacoes com erro ortografico comum (SUPRESSED).
        self.suppressed_markers = [
            'FRAGMENT_SUPPRESSED',
            'FRAGMENT_SUPRESSED',
            'CITATION_SUPPRESSED',
            'CITATION_SUPRESSED',
            'REFERENCE_SUPPRESSED',
            'REFERENCE_SUPRESSED',
        ]

        # Padrao para identificar numeros de paragrafos
        self.paragraph_pattern = r'\[(\d{1,4})\]'

    def is_french(self, text: str) -> bool:
        """
        Detecta se o texto esta em frances.

        Args:
            text: Texto a verificar

        Returns:
            True se o texto esta em frances, False caso contrario
        """
        if not text or len(text.strip()) < 10:
            return False

        try:
            lang = detect(text)
            return lang == 'fr'
        except LangDetectException:
            return False

    def _normalize_text(self, document_text: str) -> str:
        processed = document_text.replace('\n', ' ').replace('•', ' ')
        processed = re.sub(r'\s+', ' ', processed).strip()
        return processed

    def extract_all_paragraphs(self, document_text: str) -> List[Tuple[str, str]]:
        """
        Extrai todos os paragrafos do documento.

        Args:
            document_text: Texto completo do documento

        Returns:
            Lista de tuplas (id_paragrafo, texto_paragrafo)
        """
        processed = self._normalize_text(document_text)
        paragraph_matches = list(re.finditer(self.paragraph_pattern, processed))
        if not paragraph_matches:
            return []

        results: List[Tuple[str, str]] = []
        for i, match in enumerate(paragraph_matches):
            para_id = match.group(1)
            start = match.start()
            if i + 1 < len(paragraph_matches):
                end = paragraph_matches[i + 1].start()
            else:
                end = len(processed)
            para_text = processed[start:end].strip()
            results.append((para_id, para_text))
        return results

    def extract_paragraphs_with_markers(self, document_text: str) -> List[Tuple[str, str, str]]:
        """
        Extrai paragrafos que contem demarcadores e seus subsequentes.

        Args:
            document_text: Texto completo do documento

        Returns:
            Lista de tuplas (id_paragrafo, texto_paragrafo, texto_subsequente)
        """
        processed = self._normalize_text(document_text)
        paragraph_matches = list(re.finditer(self.paragraph_pattern, processed))

        if not paragraph_matches:
            return []

        results = []
        markers_upper = [m.upper() for m in self.suppressed_markers]
        for i, match in enumerate(paragraph_matches):
            para_id = match.group(1)
            start = match.start()

            if i + 1 < len(paragraph_matches):
                end = paragraph_matches[i + 1].start()
            else:
                end = len(processed)

            para_text = processed[start:end].strip()
            para_upper = para_text.upper()
            has_marker = any(marker in para_upper for marker in markers_upper)

            if has_marker:
                subsequent_text = ""
                if i + 1 < len(paragraph_matches):
                    next_start = paragraph_matches[i + 1].start()
                    if i + 2 < len(paragraph_matches):
                        next_end = paragraph_matches[i + 2].start()
                    else:
                        next_end = len(processed)

                    subsequent_text = processed[next_start:next_end].strip()

                results.append((para_id, para_text, subsequent_text))

        return results

    def clean_paragraph(self, text: str) -> str:
        """
        Remove demarcadores e formatacao desnecessaria.

        Args:
            text: Texto a limpar

        Returns:
            Texto limpo
        """
        for marker in self.suppressed_markers:
            text = re.sub(rf'<{marker}>', ' ', text, flags=re.IGNORECASE)
            text = re.sub(marker, ' ', text, flags=re.IGNORECASE)

        text = re.sub(self.paragraph_pattern, ' ', text)
        text = re.sub(r'[<>]', ' ', text)
        text = re.sub(r'\[End of document\]', ' ', text, flags=re.IGNORECASE)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def tokenize_and_process(self, text: str) -> Tuple[List[str], List[str]]:
        """
        Tokeniza e processa o texto.

        Args:
            text: Texto a processar

        Returns:
            Tupla (tokens_limpos, tokens_stemmed/lemmatized)
        """
        if self.use_spacy:
            doc = nlp(text.lower())

            tokens = [
                token.text for token in doc
                if token.is_alpha and
                   token.text not in self.stop_words and
                   len(token.text) > 2
            ]

            lemmatized = [
                token.lemma_ for token in doc
                if token.is_alpha and
                   token.lemma_ not in self.stop_words and
                   len(token.lemma_) > 2
            ]

            return tokens, lemmatized
        else:
            tokens = word_tokenize(text.lower())
            tokens = [
                token for token in tokens
                if token.isalpha() and
                   token not in self.stop_words and
                   len(token) > 2
            ]
            stemmed = [self.stemmer.stem(token) for token in tokens]
            return tokens, stemmed

    def identify_markers(self, text: str) -> List[str]:
        """
        Identifica quais demarcadores estao presentes no texto.

        Args:
            text: Texto a analisar

        Returns:
            Lista de demarcadores encontrados
        """
        found_markers = []
        upper_text = text.upper()
        for marker in self.suppressed_markers:
            if marker.upper() in upper_text or f'<{marker}>' in upper_text:
                found_markers.append(marker)
        return found_markers

    def process_document(self, doc_id: str, document_text: str, filter_french: bool = True, only_marked: bool = True) -> List[ProcessedParagraph]:
        """
        Processa o documento completo.

        Args:
            document_text: Texto completo do documento
            only_marked: Se True, usa apenas paragrafos com demarcadores

        Returns:
            Lista de paragrafos processados
        """
        if only_marked:
            paragraphs_with_markers = self.extract_paragraphs_with_markers(document_text)
        else:
            paragraphs_all = self.extract_all_paragraphs(document_text)
            paragraphs_with_markers = [(pid, ptext, "") for pid, ptext in paragraphs_all]

        processed_paragraphs = []

        for para_id, para_text, subsequent_text in paragraphs_with_markers:
            combined_text = f"{para_text} {subsequent_text}".strip()
            is_french_text = self.is_french(combined_text)
            if filter_french and is_french_text:
                continue

            cleaned = self.clean_paragraph(combined_text)
            if not cleaned or len(cleaned) < 10:
                continue

            tokens, processed_tokens = self.tokenize_and_process(cleaned)
            markers = self.identify_markers(para_text) if only_marked else []
            has_marker = len(markers) > 0
            processed_paragraphs.append(ProcessedParagraph(
                doc_id=doc_id, paragraph_id=para_id, original_text=combined_text,
                cleaned_text=cleaned, tokens=tokens, processed_tokens=processed_tokens,
                has_suppressed_marker=has_marker, marker_types=markers, is_french=is_french_text
            ))

        return processed_paragraphs

In [ ]:
class BM25S():
    def __init__(self, preprocessor: LegalDocumentPreprocessor):
        self.preprocessor = preprocessor
        self.query_paragraphs: List[ProcessedParagraph] = []
        self.corpus_paragraphs: List[ProcessedParagraph] = []
        self.corpus_texts: List[str] = []
        self.corpus_ids: List[str] = []
        self.corpus_paragraphs_info: Dict = {}

    def _serialize_paragraphs(self, paragraphs: List[ProcessedParagraph]) -> List[Dict]:
        serialized = []
        for para in paragraphs:
            serialized.append({
                'doc_id': para.doc_id,
                'paragraph_id': para.paragraph_id,
                'original_text': para.original_text,
                'cleaned_text': para.cleaned_text,
                'tokens': para.tokens,
                'processed_tokens': para.processed_tokens,
                'has_suppressed_marker': para.has_suppressed_marker,
                'marker_types': para.marker_types,
                'is_french': para.is_french,
            })
        return serialized

    def _deserialize_paragraphs(self, items: List[Dict]) -> List[ProcessedParagraph]:
        paragraphs = []
        for item in items:
            paragraphs.append(ProcessedParagraph(
                doc_id=item['doc_id'],
                paragraph_id=item['paragraph_id'],
                original_text=item.get('original_text', ''),
                cleaned_text=item.get('cleaned_text', ''),
                tokens=item.get('tokens', []),
                processed_tokens=item.get('processed_tokens', []),
                has_suppressed_marker=item.get('has_suppressed_marker', False),
                marker_types=item.get('marker_types', []),
                is_french=item.get('is_french', False),
            ))
        return paragraphs

    def _normalize_doc_id(self, doc_id: str) -> str:
        doc_id = doc_id.strip()
        if doc_id.lower().endswith('.txt'):
            return doc_id[:-4]
        return doc_id

    def _extract_result_index(self, result_item) -> Optional[int]:
        if isinstance(result_item, (int, np.integer)):
            return int(result_item)
        if isinstance(result_item, dict) and 'id' in result_item:
            return int(result_item['id'])
        return None

    def build_query_and_corpus_from_labels(self, documents_folder: str, labels_file: str, cache_dir: str,
                                           max_docs: Optional[int] = None, verbose: bool = True,
                                           filter_french: bool = True) -> Tuple[List[ProcessedParagraph], List[ProcessedParagraph]]:
        cache_path = Path(cache_dir)
        cache_path.mkdir(parents=True, exist_ok=True)
        query_cache = cache_path / 'query_paragraphs_all.json'
        corpus_cache = cache_path / 'corpus_paragraphs_all.json'

        if query_cache.exists() and corpus_cache.exists():
            if verbose:
                print("Carregando estruturas do cache...")
            with open(query_cache, 'r', encoding='utf-8') as f:
                query_items = json.load(f)
            with open(corpus_cache, 'r', encoding='utf-8') as f:
                corpus_items = json.load(f)
            self.query_paragraphs = self._deserialize_paragraphs(query_items)
            self.corpus_paragraphs = self._deserialize_paragraphs(corpus_items)
            return self.query_paragraphs, self.corpus_paragraphs

        with open(labels_file, 'r', encoding='utf-8') as f:
            labels = json.load(f)

        query_ids = {self._normalize_doc_id(k) for k in labels.keys()}
        noticed_ids = set()
        for _, values in labels.items():
            for value in values:
                noticed_ids.add(self._normalize_doc_id(value))

        if verbose:
            print(f"Query IDs: {len(query_ids)} | Noticed IDs: {len(noticed_ids)}")

        folder = Path(documents_folder)
        if not folder.exists():
            raise ValueError(f"Pasta '{documents_folder}' nao encontrada.")
        files = sorted(list(folder.glob('*.txt')))
        if max_docs:
            files = files[:max_docs]
        if verbose:
            print(f"Processando {len(files)} arquivos em '{documents_folder}'...")
        iterator = tqdm(files, desc="Processando documentos") if verbose else files

        self.query_paragraphs = []
        self.corpus_paragraphs = []
        for file_path in iterator:
            doc_id = file_path.stem
            doc_id_norm = self._normalize_doc_id(doc_id)
            if doc_id_norm not in query_ids and doc_id_norm not in noticed_ids:
                continue
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()

                # Query base: todos os paragrafos do documento com a mesma pipeline.
                if doc_id_norm in query_ids:
                    paragraphs = self.preprocessor.process_document(
                        doc_id, content, filter_french=filter_french, only_marked=False
                    )
                    self.query_paragraphs.extend(paragraphs)

                # Corpus para indice: documentos completos (agregados na indexacao).
                if doc_id_norm in noticed_ids:
                    paragraphs = self.preprocessor.process_document(
                        doc_id, content, filter_french=filter_french, only_marked=False
                    )
                    self.corpus_paragraphs.extend(paragraphs)
            except Exception as e:
                if verbose:
                    print(f"Falha ao processar {file_path}: {e}")

        with open(query_cache, 'w', encoding='utf-8') as f:
            json.dump(self._serialize_paragraphs(self.query_paragraphs), f, ensure_ascii=True, indent=2)
        with open(corpus_cache, 'w', encoding='utf-8') as f:
            json.dump(self._serialize_paragraphs(self.corpus_paragraphs), f, ensure_ascii=True, indent=2)

        return self.query_paragraphs, self.corpus_paragraphs

    def build_bm25_index_from_corpus(self, corpus_paragraphs: List[ProcessedParagraph],
                                     method: str = "lucene", k1: float = 1.5, b: float = 0.75,
                                     verbose: bool = True) -> bm25s.BM25:
        if not corpus_paragraphs:
            raise ValueError("Nenhum paragrafo no corpus")
        if verbose:
            print("Construindo indice BM25 agrupando por documento...")

        # Agrupar paragrafos por documento para indexar documentos completos.
        docs_dict = {}
        for para in corpus_paragraphs:
            if para.doc_id not in docs_dict:
                docs_dict[para.doc_id] = {
                    'tokens': [],
                    'texts': [],
                    'paragraphs_info': []
                }
            docs_dict[para.doc_id]['tokens'].extend(para.processed_tokens)
            docs_dict[para.doc_id]['texts'].append(para.cleaned_text)
            docs_dict[para.doc_id]['paragraphs_info'].append({
                'paragraph_id': para.paragraph_id,
                'marker_types': para.marker_types,
                'is_french': para.is_french,
                'token_count': len(para.processed_tokens)
            })

        self.corpus_ids = list(docs_dict.keys())
        corpus_tokens = [docs_dict[doc_id]['tokens'] for doc_id in self.corpus_ids]
        self.corpus_texts = [' '.join(docs_dict[doc_id]['texts']) for doc_id in self.corpus_ids]
        self.corpus_paragraphs_info = {doc_id: docs_dict[doc_id]['paragraphs_info'] for doc_id in self.corpus_ids}

        retriever = bm25s.BM25(method=method, k1=k1, b=b)
        retriever.index(corpus_tokens)

        if verbose:
            print(f"Indice criado! Método: {method}, k1={k1}, b={b}")
            print(f"Total de documentos no corpus: {len(self.corpus_ids)}")

        self.retriever = retriever
        return retriever

    def retrieve_documents_from_query_paragraphs(self,
                                                query_paragraphs: List[ProcessedParagraph],
                                                k_per_subquery: int = 100,
                                                top_n: int = 100,
                                                verbose: bool = True) -> List[Tuple[str, float]]:
        if not hasattr(self, 'retriever'):
            raise ValueError("Indice nao construido")

        aggregated_scores: Dict[str, float] = {}
        used_subqueries = 0

        for para in query_paragraphs:
            if not para.processed_tokens:
                continue

            results, scores = self.retriever.retrieve(para.processed_tokens, k=k_per_subquery)
            used_subqueries += 1

            for i in range(results.shape[1]):
                idx = self._extract_result_index(results[0, i])
                if idx is None or idx < 0 or idx >= len(self.corpus_ids):
                    continue

                doc_id = self.corpus_ids[idx]
                score = float(scores[0, i])
                aggregated_scores[doc_id] = aggregated_scores.get(doc_id, 0.0) + score

        ranked_docs = sorted(
            aggregated_scores.items(),
            key=lambda item: item[1],
            reverse=True
        )[:top_n]

        if verbose:
            print(f"Subqueries utilizadas: {used_subqueries}")
            print(f"Documentos pontuados: {len(aggregated_scores)}")
            print(f"Retornando Top {len(ranked_docs)} documentos por score agregado")

        return ranked_docs

    def save_index(self, save_dir: str, save_corpus: bool = True,
                   save_metadata: bool = True, verbose: bool = True):
        if not hasattr(self, 'retriever'):
            raise ValueError("Indice nao construido")

        save_path = Path(save_dir)
        save_path.mkdir(parents=True, exist_ok=True)

        if save_corpus:
            self.retriever.save(save_dir, corpus=self.corpus_texts)
        else:
            self.retriever.save(save_dir)

        if save_metadata:
            metadata = {
                'corpus_ids': self.corpus_ids,
                'total_documents': len(self.corpus_ids),
                'metadata': [
                    {'doc_id': doc_id,
                     'paragraphs_info': self.corpus_paragraphs_info.get(doc_id, []),
                     'num_paragraphs': len(self.corpus_paragraphs_info.get(doc_id, []))}
                    for doc_id in self.corpus_ids
                ],
            }
            with open(save_path / 'metadata.json', 'w', encoding='utf-8') as f:
                json.dump(metadata, f, ensure_ascii=True, indent=2)

        if verbose:
            print(f"Indice salvo em: {save_dir}")
            print(f"Total de documentos salvos: {len(self.corpus_ids)}")

## Configurar Caminhos e Parâmetros

In [ ]:
BASE_PATH = '/content/drive/MyDrive/tcc/data/'

TRAIN_FILES = os.path.join(BASE_PATH, 'task1_train_files_2025/task2025_train')
TRAIN_LABELS_FILE = os.path.join(BASE_PATH, 'task1_train_labels_2025.json')

CACHE_DIR = os.path.join(BASE_PATH, 'bm25/bm25-all-paragraphs-returns-document-id/cache')
INDEX_OUTPUT_DIR = os.path.join(BASE_PATH, 'bm25/bm25-all-paragraphs-returns-document-id/index')
MAX_DOCUMENTS = None

BM25_METHOD = "bm25l"
BM25_K1 = 1.2
BM25_B = 0.4

## Processar Documentos e Criar Índice

In [ ]:
preprocessor = LegalDocumentPreprocessor()
bm25 = BM25S(preprocessor)

In [ ]:
query_paragraphs, corpus_paragraphs = bm25.build_query_and_corpus_from_labels(
    TRAIN_FILES, TRAIN_LABELS_FILE, CACHE_DIR, max_docs=MAX_DOCUMENTS, verbose=True,
    filter_french=True
)
retriever = bm25.build_bm25_index_from_corpus(
    corpus_paragraphs, method=BM25_METHOD, k1=BM25_K1, b=BM25_B, verbose=True
)

bm25.save_index(save_dir=INDEX_OUTPUT_DIR, save_corpus=True, save_metadata=True, verbose=True)

Query IDs: 1678 | Noticed IDs: 5529
Processando 7350 arquivos em '/content/drive/MyDrive/tcc/data/task1_train_files_2025/task2025_train'...


Processando documentos: 100%|██████████| 7350/7350 [1:37:59<00:00,  1.25it/s]
DEBUG:bm25s:Building index from tokens


Construindo indice BM25 apenas do corpus...


BM25S Create Vocab:   0%|          | 0/239212 [00:00<?, ?it/s]

BM25S Convert tokens to indices:   0%|          | 0/239212 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/239212 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/239212 [00:00<?, ?it/s]

Indice criado! Metodo: bm25l, k1=1.2, b=0.4


Finding newlines for mmindex:   0%|          | 0.00/139M [00:00<?, ?B/s]

Indice salvo em: /content/drive/MyDrive/tcc/data/bm25/bm25l2-0/index


##Testar o Índice - Fazer Buscas

In [ ]:
corpus_ids = bm25.corpus_ids
print(f"Metadados carregados ({len(corpus_ids)} documentos no corpus)")

Metadados carregados (95147 documentos)


In [ ]:
# Fazer uma busca de teste usando cada paragrafo do documento de query como subquery
if not query_paragraphs:
    raise ValueError("Nenhum paragrafo na estrutura QUERY")

query_doc_id = query_paragraphs[0].doc_id
query_doc_paragraphs = [p for p in query_paragraphs if p.doc_id == query_doc_id]

if not query_doc_paragraphs:
    raise ValueError(f"Nenhum paragrafo encontrado para o documento de query: {query_doc_id}")

print(f"Query doc_id: {query_doc_id}")
print(f"Total de paragrafos (subqueries): {len(query_doc_paragraphs)}")

top_docs = bm25.retrieve_documents_from_query_paragraphs(
    query_paragraphs=query_doc_paragraphs,
    k_per_subquery=100,
    top_n=100,
    verbose=True
)

print("\n" + "="*80)
print("RESULTADOS DA BUSCA (TOP 100 DOCUMENTOS POR SCORE AGREGADO)")
print("="*80)

for rank, (doc_id, agg_score) in enumerate(top_docs, start=1):
    print(f"\n[Rank {rank}] Score agregado: {agg_score:.4f}")
    print(f"ID do Documento: {doc_id}")
    print("-"*80)

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Query: 'procedural fairness investigation'
Tokens: Tokenized(
  "ids": [
    0: [0, 1, 2]
  ],
  "vocab": [
    'fairness': 1
    'investigation': 2
    'procedural': 0
  ],
)


NameError: name 'retriever' is not defined

## (Opcional) Recarregar Índice em Outra Sessão

In [ ]:
%pip install "bm25s[full]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.9/88.9 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.3/745.3 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 9.0 MB/s eta 0:00:00
  Created wheel for pytrec_eval: filename=pytrec_eval-0.5-cp312-cp312-linux_x86_64.whl size=309354 sha256=54f5e5851b2f850bf32d92846478ebbc4aff72fa0891937afc14f0cb186e0325
  Stored in directory: /root/.cache/pip/wheels/c6/4a/9e/e17f9ea004e1c221bd0ff384732285211c4917b790d598ea51
Successfully built pytrec_eval


In [ ]:
import bm25s
import json
import os

from google.colab import drive

# Montar Drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Se voce quiser recarregar o indice em outra sessao:
BASE_PATH = '/content/drive/MyDrive/tcc/data/'

INDEX_DIR = os.path.join(BASE_PATH, 'bm25/bm25-full-document-query/index/')
CACHE_DIR = os.path.join(BASE_PATH, 'bm25/bm25-full-document-query/cache/')

loaded_retriever = bm25s.BM25.load(INDEX_DIR, load_corpus=True)

with open(f"{INDEX_DIR}/metadata.json", 'r') as f:
    loaded_metadata = json.load(f)

with open(os.path.join(CACHE_DIR, 'query_paragraphs_all.json'), 'r') as f:
    query_items = json.load(f)

corpus_ids = loaded_metadata['corpus_ids']
print("Indice e cache recarregados!")
print(f"Documentos no corpus: {loaded_metadata['total_documents']}")

In [ ]:
# Exemplos de queries customizadas
# first_item = query_items[0]

# query = first_item['cleaned_text']
# query_with_labels = "foreign law question of fact expert opinion evidence own interpretation substituted understanding reasonableness standard review visa officer disregarded"
# expanded_window_query = "interpretation foreign law questions of fact standard reasonableness substitute own opinion set aside legal opinions"
# query_with_paragraphs_markers = "interpretation foreign law questions of fact standard of reasonableness substitute own opinion decision-maker"
# blind_query_gemini = ["procedural fairness", "standard of correctness", "Cameroonian law", "questions of fact", "standard of reasonableness", "deferential", "substitute its own opinion",
#                       "justification, transparency and intelligibility", "possible, acceptable outcomes", "defensible in respect of the facts and law", "no breach of procedural fairness",
#                       "opportunity to submit evidence and arguments"]
# blind_query_gemini = "procedural fairness standard of correctness Cameroonian law questions of fact standard of reasonableness deferential substitute its own opinion justification transparency and intelligibility possible acceptable outcomes defensible in respect of the facts and law no breach of procedural fairness opportunity to submit evidence and arguments"

# blind_query_gpt1 = "procedural fairness duty of fairness natural justice right to be heard hearing before decision immigration appeal division administrative tribunal judicial review standard of review correctness reasonableness administrative decision justification transparency intelligibility decision making process acceptable outcomes failure to hold hearing"
# blind_query_gpt2 = "procedural fairness duty of fairness natural justice right to be heard hearing before decision judicial review standard of review reasonableness correctness administrative decision justification transparency intelligibility acceptable outcomes immigration appeal division administrative tribunal"
# blind_query_gpt3 = "judicial review reasonableness standard procedural fairness duty of fairness natural justice administrative decision justification transparency intelligibility acceptable outcomes decision making process administrative tribunal immigration appeal division hearing fairness"

blind_query_gpt3 = "judicial review reasonableness standard procedural fairness duty of fairness natural justice administrative decision justification transparency intelligibility acceptable outcomes decision making process administrative tribunal immigration appeal division hearing fairness"
query_tokens = bm25s.tokenize(blind_query_gpt3, stopwords="en")

results, scores = loaded_retriever.retrieve(query_tokens, k=300)

print("\n" + "="*80)
print("RESULTADOS DA BUSCA - TOP 100 DOCUMENTOS")
print("="*80)

# Rastrear documentos únicos para retornar apenas 100 documentos
unique_docs = {}
rank = 1
for i in range(results.shape[1]):
    result = results[0, i]
    score = scores[0, i]
    idx = result['id']
    text = result['text']
    doc_id = corpus_ids[idx]
    
    # Verificar se o documento já foi incluído
    if doc_id not in unique_docs:
        unique_docs[doc_id] = {
            'score': score,
            'text': text,
            'rank': rank
        }
        rank += 1
        
        if rank <= 101:  # Mostrar até 100 documentos únicos
            print(f"\n[Rank {unique_docs[doc_id]['rank']}] Score: {score:.4f}")
            print(f"ID do Documento: {doc_id}")
            print(f"Texto: {text[:200]}...")
            print("-"*80)
    
    if rank > 101:  # Parar após encontrar 100 documentos únicos
        break

print(f"\nTotal de documentos únicos encontrados: {len(unique_docs)}")

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


RESULTADOS DA BUSCA

[Rank 1] Score: 100.9824
ID: 089334_2001
Texto: F.T.R. ) 3, 2001. Summary: The applicants applied under s. 220(3.1) of the Income Tax Act (fairness legislation) for cancellation of interest and penalties assessed against them. A Tax Services Direct...
--------------------------------------------------------------------------------

[Rank 2] Score: 100.7297
ID: 061827_9
Texto: The Board's decision regarding credibility is reviewed based on the reasonableness standard ( at para 47, reasonableness is "concerned mostly with the existence of justification, transparency and inte...
--------------------------------------------------------------------------------

[Rank 3] Score: 100.0432
ID: 004173_20
Texto: Thus, in light of the Supreme Court of Canada's decision in and the previous jurisprudence of this Court, I find the standard of review applicable to the non-procedural fairness issues in this case to...
---------------------------------------------------------------

In [ ]:
# with open(BASE_PATH + 'cases_dict.json', 'r') as f:
#   train_cases_dict = json.load(f)

# with open(TRAIN_LABELS_FILE, 'r') as f:
#   train_labels = json.load(f)

In [ ]:
# class BM25():
#   def __init__(self,
#                k1: float = 1.5,
#                b: float = 0.75,
#                delta: float = 0.5):
#     self.k1 = k1
#     self.b = b
#     self.delta = delta
#     self.stemmer = Stemmer.Stemmer("en")

#     self.retriever = None
#     self.case_ids = []
#     self.corpus = {}
#     self.tokenizer = None

#   def preprocess_document(self, paragraphs: List[str]) -> str:
#     return " ".join(paragraphs)

#   def build_index(self, cases: Dict[str, List[str]]):
#     print("Construindo indice...")

#     self.case_ids = list(cases.keys())
#     corpus_texts = []

#     for case_id in tqdm(self.case_ids, desc="Preparando docs"):
#       doc_text = self.preprocess_document(cases[case_id])
#       corpus_texts.append(doc_text)
#       self.corpus[case_id] = doc_text

#     corpus_tokens = bm25s.tokenize(
#         corpus_texts,
#         stopwords="en",
#         stemmer=self.stemmer,
#         show_progress=True
#     )

#     self.retriever = bm25s.BM25(
#         method="bm25l",
#         k1=self.k1,
#         b=self.b,
#         delta=self.delta
#     )

#     self.retriever.index(corpus_tokens)

#   def load_cases(self, cases_path: str) -> Dict[str, List[str]]:
#     print("Carregando casos...")

#     with open(cases_path, 'r', encoding='utf-8') as f:
#       cases = json.load(f)

#     return cases

#   def load_queries(self, queries_path: str) -> Dict[str, List[str]]:
#     print("Carregando queries...")

#     with open(queries_path, 'r', encoding='utf-8') as f:
#       queries = json.load(f)

#     return queries

#   def retrieve(self, query: str, k: int = 10) -> Tuple[List[str], np.ndarray]:
#     if self.retriever is None:
#       raise ValueError("Índice não construído. Chame o método 'build_index' primeiro.")

#     query_tokens = bm25s.tokenize(
#         query,
#         stopwords="en",
#         stemmer=self.stemmer
#     )

#     results, scores = self.retriever.retrieve(query_tokens, k=k)

#     retrieved_case_ids = [self.case_ids[idx] for idx in results[0]]

#     return retrieved_case_ids, scores[0]

#   def evaluate(self,
#                  query_cases: Dict[str, List[str]],
#                  cases: Dict[str, List[str]],
#                  k_values: List[int] = [1, 5, 10, 20]) -> Dict:
#         """
#         Avalia o sistema usando as queries e casos relevantes.

#         Args:
#             query_cases: Dicionário com queryId: [casos relevantes]
#             cases: Dicionário com todos os casos
#             k_values: Lista de valores de k para avaliar

#         Returns:
#             Dicionário com métricas de avaliação e detalhes
#         """
#         print("\n" + "="*70)
#         print("📊 AVALIANDO SISTEMA DE RECUPERAÇÃO")
#         print("="*70)

#         results = {
#             'precision': {k: [] for k in k_values},
#             'recall': {k: [] for k in k_values},
#             'f1': {k: [] for k in k_values},
#             'mrr': [],
#             'map': [],  # Mean Average Precision
#             'ndcg': {k: [] for k in k_values},
#             'details': []  # Para análise detalhada
#         }

#         total_queries = len(query_cases)

#         for query_id, relevant_cases in tqdm(query_cases.items(), desc="Avaliando queries"):
#             if query_id not in cases:
#                 print(f"⚠️ Query {query_id} não encontrada nos casos. Pulando...")
#                 continue

#             # Usa o caso da query como consulta
#             query_text = self.preprocess_document(cases[query_id])

#             # Recupera casos (k máximo + 1 para poder excluir o próprio caso se necessário)
#             max_k = max(k_values) + 1
#             retrieved_ids, scores = self.retrieve(query_text, k=max_k)

#             # Remove o próprio caso da query dos resultados
#             filtered_results = [(cid, score) for cid, score in zip(retrieved_ids, scores)
#                                if cid != query_id]
#             retrieved_ids = [cid for cid, _ in filtered_results[:max(k_values)]]
#             retrieved_scores = [score for _, score in filtered_results[:max(k_values)]]

#             # Calcula Average Precision para MAP
#             ap = 0
#             relevant_found = 0
#             for i, cid in enumerate(retrieved_ids):
#                 if cid in relevant_cases:
#                     relevant_found += 1
#                     precision_at_i = relevant_found / (i + 1)
#                     ap += precision_at_i
#             ap = ap / len(relevant_cases) if len(relevant_cases) > 0 else 0
#             results['map'].append(ap)

#             # Detalhes para esta query
#             query_detail = {
#                 'query_id': query_id,
#                 'num_relevant': len(relevant_cases),
#                 'retrieved_top10': retrieved_ids[:10],
#                 'scores_top10': [float(s) for s in retrieved_scores[:10]],
#                 'relevant_cases': relevant_cases,
#                 'relevant_found': [cid for cid in retrieved_ids[:20] if cid in relevant_cases]
#             }
#             results['details'].append(query_detail)

#             # Calcula métricas para cada k
#             for k in k_values:
#                 retrieved_k = retrieved_ids[:k]

#                 # Precision@k
#                 relevant_retrieved = len(set(retrieved_k) & set(relevant_cases))
#                 precision = relevant_retrieved / k if k > 0 else 0
#                 results['precision'][k].append(precision)

#                 # Recall@k
#                 recall = relevant_retrieved / len(relevant_cases) if len(relevant_cases) > 0 else 0
#                 results['recall'][k].append(recall)

#                 # F1@k
#                 f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
#                 results['f1'][k].append(f1)

#                 # NDCG@k
#                 dcg = sum([1 / np.log2(i + 2) for i, cid in enumerate(retrieved_k)
#                           if cid in relevant_cases])
#                 idcg = sum([1 / np.log2(i + 2) for i in range(min(k, len(relevant_cases)))])
#                 ndcg = dcg / idcg if idcg > 0 else 0
#                 results['ndcg'][k].append(ndcg)

#             # MRR (Mean Reciprocal Rank)
#             for i, cid in enumerate(retrieved_ids):
#                 if cid in relevant_cases:
#                     results['mrr'].append(1 / (i + 1))
#                     break
#             else:
#                 results['mrr'].append(0)

#         # Calcula médias
#         summary = {
#             'precision': {k: np.mean(v) for k, v in results['precision'].items()},
#             'recall': {k: np.mean(v) for k, v in results['recall'].items()},
#             'f1': {k: np.mean(v) for k, v in results['f1'].items()},
#             'mrr': np.mean(results['mrr']),
#             'map': np.mean(results['map']),
#             'ndcg': {k: np.mean(v) for k, v in results['ndcg'].items()},
#             'num_queries': total_queries,
#             'details': results['details']
#         }

#         # Imprime resultados
#         print("\n📈 RESULTADOS DA AVALIAÇÃO:")
#         print("="*70)
#         print(f"{'Métrica':<15} {'k=1':<10} {'k=5':<10} {'k=10':<10} {'k=20':<10}")
#         print("-"*70)
#         print(f"{'Precision@k':<15} {summary['precision'].get(1, 0):<10.4f} "
#               f"{summary['precision'].get(5, 0):<10.4f} {summary['precision'].get(10, 0):<10.4f} "
#               f"{summary['precision'].get(20, 0):<10.4f}")
#         print(f"{'Recall@k':<15} {summary['recall'].get(1, 0):<10.4f} "
#               f"{summary['recall'].get(5, 0):<10.4f} {summary['recall'].get(10, 0):<10.4f} "
#               f"{summary['recall'].get(20, 0):<10.4f}")
#         print(f"{'F1@k':<15} {summary['f1'].get(1, 0):<10.4f} "
#               f"{summary['f1'].get(5, 0):<10.4f} {summary['f1'].get(10, 0):<10.4f} "
#               f"{summary['f1'].get(20, 0):<10.4f}")
#         print(f"{'NDCG@k':<15} {summary['ndcg'].get(1, 0):<10.4f} "
#               f"{summary['ndcg'].get(5, 0):<10.4f} {summary['ndcg'].get(10, 0):<10.4f} "
#               f"{summary['ndcg'].get(20, 0):<10.4f}")
#         print("-"*70)
#         print(f"{'MRR':<15} {summary['mrr']:<10.4f}")
#         print(f"{'MAP':<15} {summary['map']:<10.4f}")
#         print("="*70)

#         return summary

In [ ]:
# drive.mount('/content/drive')
# BASE_PATH = '/content/drive/MyDrive/tcc/data/'
# TRAIN_FILES = os.path.join(BASE_PATH, 'cases_dict.json')
# TRAIN_LABELS_FILE = os.path.join(BASE_PATH, 'task1_train_labels_2025.json')

# bm25 = BM25()

# cases = bm25.load_cases(TRAIN_FILES)
# queries = bm25.load_queries(TRAIN_LABELS_FILE)

# bm25.build_index(cases)
# # results = bm25.evaluate(queries, cases, k_values=[1, 5, 10, 20])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Carregando casos...
Carregando queries...
Construindo indice...


Preparando docs: 100%|██████████| 7350/7350 [00:00<00:00, 47230.84it/s]


Split strings:   0%|          | 0/7350 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/7350 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects


BM25S Count Tokens:   0%|          | 0/7350 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/7350 [00:00<?, ?it/s]

In [ ]:
# query_id = list(queries.keys())[0]
# query_text = bm25.preprocess_document(cases[query_id])

# retrieved, scores = bm25.retrieve(query_text, k=5)

# print(f"\nQuery: {query_id}")
# print(f"Casos relevantes esperados: {queries[query_id][:3]}")
# print("\nCasos recuperados:")
# for i, (case_id, score) in enumerate(zip(retrieved, scores), 1):
#   print(f"{i}. {case_id} (score: {score:.4f})")


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Query: 008447.txt
Casos relevantes esperados: ['072495.txt', '082291.txt', '004851.txt']

Casos recuperados:
1. 008447.txt (score: 4114.9082)
2. 089987.txt (score: 4059.3633)
3. 052325.txt (score: 2488.9790)
4. 031557.txt (score: 2482.9534)
5. 069049.txt (score: 2472.5796)
